# Teste Final do Pipeline Multimodal com LangGraph + AWS S3

Este notebook realiza a validação completa do pipeline multimodal desenvolvido para apoio à triagem clínica preventiva em saúde feminina.

O fluxo integra:
- upload de arquivos multimodais no Amazon S3;
- download automático pelo LangGraph;
- processamento de vídeo;
- processamento de áudio;
- fusão multimodal;
- geração de resposta interpretativa com LLM.

O pipeline utiliza:
- AWS S3;
- AWS Rekognition;
- YOLOv8;
- OpenCV;
- Librosa;
- LangGraph;
- RAG;
- Mistral via Ollama.

Os testes utilizam arquivos da base pública RAVDESS, composta por atores e emoções simuladas, evitando uso de dados reais de pacientes.

A análise possui finalidade exclusivamente educacional e de apoio à triagem preventiva.

In [1]:
import sys
print(sys.executable)

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\venv\Scripts\python.exe


In [2]:
!{sys.executable} -m pip install moviepy imageio imageio-ffmpeg


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!{sys.executable} -m pip show moviepy

Name: moviepy
Version: 2.2.1
Summary: Video editing with Python
Home-page: 
Author: Zulko 2024
Author-email: 
License: MIT License
Location: C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\venv\Lib\site-packages
Requires: decorator, imageio, imageio_ffmpeg, numpy, pillow, proglog, python-dotenv
Required-by: 


In [4]:
from moviepy import VideoFileClip

print("MoviePy funcionando.")

MoviePy funcionando.


In [5]:
import sys
sys.path.append("..")

import os
import boto3

from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv

from src.workflows.langgraph_flow import (
    build_medical_assistant_graph
)

from src.llm.ollama_client import (
    get_llm
)

from src.rag.vector_store import (
    load_vector_store
)

from src.rag.retriever import (
    get_retriever,
    retrieve_context
)

from src.multimodal.media_utils import (
    extract_audio_from_video
)

In [6]:
llm = get_llm(
    model_name="mistral",
    temperature=0.2
)

vector_store = load_vector_store(
    "../data/vector_store"
)

retriever = get_retriever(
    vector_store,
    k=3
)

app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever,
    db_path="../data/medical_demo.db"
)

print("LangGraph compilado com sucesso.")

C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\notebooks\..\src\llm\ollama_client.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  return ChatOllama(


Base vetorial carregada de: ../data/vector_store
LangGraph compilado com sucesso.


### CONFIGURAÇÃO AWS
Nesta etapa são carregadas as variáveis de ambiente e a configuração do Amazon S3 utilizada para armazenamento temporário dos arquivos multimodais.

In [7]:
load_dotenv()

bucket = os.getenv("AWS_S3_BUCKET")
region = os.getenv("AWS_REGION")

print("Bucket:", bucket)
print("Region:", region)

Bucket: postechfase4
Region: us-east-1


In [8]:
s3 = boto3.client(
    "s3",
    region_name=region
)

print("Cliente S3 criado com sucesso.")

Cliente S3 criado com sucesso.



### ESCOLHA DO CENÁRIO

Nesta etapa é selecionado o cenário multimodal utilizado para teste.

Os arquivos pertencem à base pública RAVDESS e representam emoções simuladas por atores.

Cenários sugeridos:
- calm
- sad
- fearful
- angry

In [9]:
video = r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\video_calm.mp4"

print("Vídeo existe?", os.path.exists(video))
print(video)

Vídeo existe? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\video\video_calm.mp4


### CÉLULA 8 - EXTRAÇÃO DO ÁUDIO

O áudio é extraído automaticamente do vídeo para permitir análise acústica e textual da fala.

In [12]:
audio_output = Path(
    r"C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\extract_calm_audio_female.wav"
)

extract_audio_from_video(
    video_path=str(video),
    output_audio_path=str(audio_output)
)

print("Áudio extraído com sucesso.")

Áudio extraído com sucesso.


In [14]:

print("Áudio encontrado?", audio_output.exists())
print(audio_output)

Áudio encontrado? True
C:\Users\keity\OneDrive\Documentos\GitHub\postech\tech-challenge-ia-diagnostico\tech-challenge-ia-diagnostico\data\multimodal\audio\extract_calm_audio_female.wav


### CÉLULA 11 - UPLOAD S3

Nesta etapa os arquivos multimodais são enviados para o Amazon S3.

O pipeline LangGraph realizará o download automático durante a execução.

In [18]:
video_s3_key = (
    "inputs/videos/video_calm.mp4"
)

audio_s3_key = (
    "inputs/audios/extract_calm_audio_female.wav"
)

s3.upload_file(
    str(video),
    bucket,
    video_s3_key
)

print("Vídeo enviado para o S3.")

Vídeo enviado para o S3.


In [19]:
s3.upload_file(
    str(audio_output),
    bucket,
    audio_s3_key
)

print("Áudio enviado para o S3.")

Áudio enviado para o S3.


In [20]:
for key in [video_s3_key, audio_s3_key]:

    response = s3.head_object(
        Bucket=bucket,
        Key=key
    )

    print(
        key,
        "-",
        response["ContentLength"],
        "bytes"
    )

inputs/videos/video_calm.mp4 - 4534053 bytes
inputs/audios/extract_calm_audio_female.wav - 629826 bytes


### CÉLULA 16 - BUILD LANGGRAPH

Nesta etapa o LangGraph é compilado para execução do pipeline multimodal integrado.

In [21]:
    app = build_medical_assistant_graph(
    llm=llm,
    retriever=retriever
)

print("LangGraph compilado com sucesso.")

LangGraph compilado com sucesso.


### CÉLULA 18 - CRIAÇÃO DO ESTADO

O estado contém:
- pergunta do usuário;
- chaves dos arquivos multimodais no S3;
- idioma do áudio;
- identificação do cenário.

In [22]:
state = {

    "question": (
        "Avalie possíveis sinais de risco "
        "emocional na análise multimodal."
    ),

    "audio_s3_key": audio_s3_key,

    "video_s3_key": video_s3_key,

    "audio_language": "en-US",

    "patient_id": "RAVDESS_FEARFUL_01"
}

### CÉLULA 20 - EXECUÇÃO DO PIPELINE

O pipeline executa automaticamente:
- download dos arquivos do S3;
- análise de vídeo;
- análise de áudio;
- fusão multimodal;
- geração de resposta;
- alerta preventivo.

In [23]:
result = app.invoke(state)

print("Pipeline executado com sucesso.")

UnboundLocalError: cannot access local variable 'interpretation' where it is not associated with a value

### CÉLULA 22 - RESULTADO DO VÍDEO

Nesta etapa é exibido o resultado da análise visual.